# Notebook 1: Fine-tuning LLM (Basic)


In [3]:
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo
!pip install wandb transformers datasets peft trl


In [4]:
import os
import torch
import wandb
from google.colab import userdata
from datasets import load_dataset
from unsloth import FastLanguageModel, get_chat_template, is_bfloat16_supported
from trl import SFTTrainer
from transformers import TrainingArguments
from huggingface_hub import login

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

login(token=os.environ["HF_TOKEN"])
wandb.login(key=os.environ["WANDB_API_KEY"])


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: arigusnawawa11 (arigusnawawa11-upn-veteran-jawa-timur) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
max_seq_length = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)


==((====))==  Unsloth 2026.7.3: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.
Unsloth 2026.7.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [6]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

dataset = load_dataset("Ichsan2895/alpaca-gpt4-indonesian", split="train")

def convert_to_convo(example):
    convo = [
        {"role": "system", "content": "Anda adalah asisten AI berbahasa Indonesia yang membantu dan pintar."},
        {"role": "user", "content": example["input"]}, # Hanya gunakan example["input"] di sini
        {"role": "assistant", "content": example["output"]}
    ]
    return {"conversations": convo}

print(dataset[0])
dataset = dataset.map(convert_to_convo)
dataset = dataset.map(formatting_prompts_func, batched=True)
print(dataset[0]["text"])

README.md:   0%|          | 0.00/1.91k [00:00<?, ?B/s]

alpaca-gpt4-indonesia.csv:   0%|          | 0.00/41.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49969 [00:00<?, ? examples/s]

{'Unnamed: 0': 1, 'input': 'Saranlah slogan untuk kampanye daur ulang\n', 'output': '1. "Kurangi, gunakan kembali, daur ulang: Bersama untuk masa depan yang lebih hijau."\n2. "Daur ulanglah hari ini, untuk masa depan yang lebih baik."\n3. "Ubah sampahmu menjadi harta karun - Daur ulang!"\n4. "Daur ulang untuk siklus kehidupan."\n5. "Simpan sumber daya, daur ulang lebih banyak."'}


Map:   0%|          | 0/49969 [00:00<?, ? examples/s]

Map:   0%|          | 0/49969 [00:00<?, ? examples/s]

<|im_start|>system
Anda adalah asisten AI berbahasa Indonesia yang membantu dan pintar.<|im_end|>
<|im_start|>user
Saranlah slogan untuk kampanye daur ulang
<|im_end|>
<|im_start|>assistant
1. "Kurangi, gunakan kembali, daur ulang: Bersama untuk masa depan yang lebih hijau."
2. "Daur ulanglah hari ini, untuk masa depan yang lebih baik."
3. "Ubah sampahmu menjadi harta karun - Daur ulang!"
4. "Daur ulang untuk siklus kehidupan."
5. "Simpan sumber daya, daur ulang lebih banyak."<|im_end|>



In [7]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",      # <-- PINDAH KE DALAM SINI
        max_seq_length=max_seq_length,  # <-- PINDAH KE DALAM SINI
        dataset_num_proc=2,             # <-- PINDAH KE DALAM SINI
        packing=True,                   # <-- PINDAH KE DALAM SINI

        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=800,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        output_dir="outputs_basic",
        report_to="wandb",
    ),
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/49969 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=2):   0%|          | 0/49969 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 26,109 | Num Epochs = 1 | Total steps = 800
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.639752
20,1.084803
30,1.042436
40,1.004671
50,1.012301
60,0.996666
70,1.040538
80,0.998336
90,0.997093
100,0.975361


Unsloth: Restored added_tokens_decoder metadata in outputs_basic/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_basic/checkpoint-800/tokenizer_config.json.


TrainOutput(global_step=800, training_loss=0.9633576810359955, metrics={'train_runtime': 11494.5578, 'train_samples_per_second': 0.557, 'train_steps_per_second': 0.07, 'total_flos': 1.4609234063757312e+17, 'train_loss': 0.9633576810359955, 'epoch': 0.24511681348142475})

In [8]:
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

hf_username = "bazwa"
model_name = "llama-3-8b-indonesian-sft"

model.push_to_hub_merged(f"{hf_username}/{model_name}", tokenizer, save_method="merged_16bit", token=os.environ["HF_TOKEN"])


Unsloth: Restored added_tokens_decoder metadata in lora_model/tokenizer_config.json.


config.json:   0%|          | 0.00/768 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in bazwa/llama-3-8b-indonesian-sft/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0004-of-00004.safetensors:   1%|1         | 15.1MB / 1.17GB            


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [08:01<00:00, 120.31s/it]


Unsloth: Merge process complete. Saved to `/content/bazwa/llama-3-8b-indonesian-sft`
